# Micro-Learning Module Generation - Testing Notebook

This notebook tests the generation of comprehensive micro-learning modules from the RAG vector store.

**Key Features:**
- Category-based content retrieval from ChromaDB
- OpenAI embeddings (`text-embedding-3-small`)
- Multi-chapter structured learning with Groq LLM
- Detailed, extensive micro-content (NOT one-liners)
- JSON format ready for Quickbase integration

**Prerequisites:**
- Vector store created via `01_data_ingestion_vector_store.ipynb`
- Environment variables: `OPENAI_API_KEY`, `GROQ_API_KEY`

In [9]:

import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Vector store and embeddings
import chromadb
from chromadb.utils import embedding_functions

# LLM
from groq import Groq

# Utilities
import json
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv

# Load environment variables
env_path = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\.env")
load_dotenv(env_path)

print("✅ All libraries imported successfully")
print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All libraries imported successfully
⏰ Started at: 2026-04-07 19:25:50


## Configuration and Setup

In [10]:

import csv

# Paths
BASE_DIR = Path(r"C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF")
VECTOR_STORE_PATH = BASE_DIR / "vector_store"
DATA_DIR = BASE_DIR / "data"
CATEGORY_FILE_PATH = DATA_DIR / "category_file" / "categories.csv"

# API Keys
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY not found in .env file")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("❌ OPENAI_API_KEY not found in .env file")

# Embedding model (must match ingestion)
EMBEDDING_MODEL = "text-embedding-3-small"  # OpenAI model

# LLM model
LLM_MODEL = "llama-3.3-70b-versatile"

# Retrieval configuration
TOP_K_CHUNKS = 20  # Retrieve more chunks for comprehensive content

# Load category to course_id mapping from CSV (same as ingestion notebook)
def normalize_category_name(category_name: str) -> str:
    """Convert category display name to folder name format."""
    return category_name.lower().replace(' & ', '_and_').replace(' ', '_')

def load_category_mappings():
    """Load category to course_id mappings from CSV."""
    category_to_course_id = {}
    categories = []
    
    if not CATEGORY_FILE_PATH.exists():
        raise FileNotFoundError(f"categories.csv not found at {CATEGORY_FILE_PATH}")
    
    with open(CATEGORY_FILE_PATH, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if row.get('active', '').lower() == 'true':
                category_display_name = row.get('category_name', '').strip()
                course_id = row.get('course_id', '').strip()
                
                if category_display_name and course_id:
                    category_folder_name = normalize_category_name(category_display_name)
                    categories.append(category_folder_name)
                    category_to_course_id[category_folder_name] = course_id
    
    return categories, category_to_course_id

# Load categories and mappings
CATEGORIES, CATEGORY_TO_COURSE_ID = load_category_mappings()

print(f"✅ Configuration loaded")
print(f"💾 Vector Store: {VECTOR_STORE_PATH}")
print(f"🤖 Embedding Model: {EMBEDDING_MODEL} (OpenAI)")
print(f"🤖 LLM Model: {LLM_MODEL}")
print(f"📊 Top-K Retrieval: {TOP_K_CHUNKS}")
print(f"📂 Categories: {CATEGORIES}")
print(f"📋 Category → Course ID: {CATEGORY_TO_COURSE_ID}")

✅ Configuration loaded
💾 Vector Store: C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\vector_store
🤖 Embedding Model: text-embedding-3-small (OpenAI)
🤖 LLM Model: llama-3.3-70b-versatile
📊 Top-K Retrieval: 20
📂 Categories: ['sustainable_agriculture_and_natural_resources', 'sustainable_agriculture_and_natural_resources', 'sustainability_strategy_and_compliance', 'circular_economy_and_waste_reduction', 'circular_economy_and_waste_reduction']
📋 Category → Course ID: {'sustainable_agriculture_and_natural_resources': '003', 'sustainability_strategy_and_compliance': '002', 'circular_economy_and_waste_reduction': '001'}


## Step 1: Initialize Vector Store and Models

Connect to the vector store created in the ingestion notebook and initialize:
- **OpenAI embedding function** (must match the model used during ingestion)
- **ChromaDB client** with persistent storage
- **Groq LLM client** for content generation

In [11]:

print("🔄 Initializing OpenAI embeddings...")
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBEDDING_MODEL
)
print(f"✅ OpenAI embedding function initialized")

print("\n🔄 Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_PATH))
collection = chroma_client.get_collection(
    name="wwf_knowledge_base",
    embedding_function=openai_ef
)
print(f"✅ Connected to collection: wwf_knowledge_base")
print(f"   Total chunks: {collection.count():,}")

print("\n🔄 Initializing Groq client...")
groq_client = Groq(api_key=GROQ_API_KEY)
print(f"✅ Groq client initialized")

🔄 Initializing OpenAI embeddings...
✅ OpenAI embedding function initialized

🔄 Connecting to ChromaDB...
✅ Connected to collection: wwf_knowledge_base
   Total chunks: 312

🔄 Initializing Groq client...
✅ Groq client initialized


## Step 2: Retrieve Relevant Content from Vector Store

In [12]:

def retrieve_category_content(category: str, top_k: int = 20) -> List[str]:
    """
    Retrieve diverse, representative content chunks for a category.
    
    Args:
        category: Category name
        top_k: Number of chunks to retrieve
        
    Returns:
        List of text chunks
    """
    print(f"\n🔍 Retrieving content for category: {category}")
    
    # Get all chunks for this category
    results = collection.get(
        where={"category": category},
        limit=top_k
    )
    
    if not results['documents']:
        print(f"❌ No content found for category: {category}")
        return []
    
    chunks = results['documents']
    metadatas = results['metadatas']
    
    print(f"✅ Retrieved {len(chunks)} chunks")
    
    # Show source distribution
    sources = {}
    for meta in metadatas:
        source = meta['source_file']
        sources[source] = sources.get(source, 0) + 1
    
    print(f"\n📊 Content distribution:")
    for source, count in sources.items():
        print(f"   📄 {source}: {count} chunks")
    
    return chunks


# Test retrieval for a category
test_category = "circular_economy_and_waste_reduction"
retrieved_chunks = retrieve_category_content(test_category, TOP_K_CHUNKS)

print(f"\n📝 Sample chunk preview:")
print(f"{'='*80}")
print(retrieved_chunks[0][:500] if retrieved_chunks else "No content")
print(f"{'='*80}")


🔍 Retrieving content for category: circular_economy_and_waste_reduction
✅ Retrieved 20 chunks

📊 Content distribution:
   📄 cewr_1.pdf: 20 chunks

📝 Sample chunk preview:
--- Page 1 ---
 
PURPOSE OF THIS DOCUMENT ................................ ................................ ................................ ............. 1 
BACKGROUND ................................ ................................ ................................ ................................ . 1 
THE BENEFITS OF REUSE ................................ ................................ ................................ .................... 1 
SOLID WASTE, POLLUTION AND EMISSIONS REDUCTIONS .


## Step 3: Generate Micro-Learning Modules with LLM

In [13]:

def generate_microlearning_modules(category: str, chunks: List[str]) -> Dict:
    """
    Generate comprehensive micro-learning modules from content chunks.
    
    Args:
        category: Category name
        chunks: List of content chunks
        
    Returns:
        Structured micro-learning JSON
    """
    print(f"\n🤖 Generating micro-learning modules for: {category}")
    
    # Get course_id from mapping
    course_id = CATEGORY_TO_COURSE_ID.get(category, "001")
    
    # Combine chunks into context
    combined_content = "\n\n---\n\n".join(chunks[:15])  # Use top 15 chunks
    
    # Format category name for display
    category_display = category.replace('_', ' ').title()
    
    # Create comprehensive prompt for ebook-style content
    prompt = f"""You are an expert instructional designer creating comprehensive micro-learning modules for sustainability education.

Based on the following WWF content about "{category_display}", create a detailed micro-learning course structure formatted like a professional ebook.

CONTENT:
{combined_content}

EBOOK CONTENT STRUCTURE REQUIREMENTS:

1. **Chapters:** Create 4-6 well-structured chapters covering key topics

2. **Content Style - Mix of Formats (like an ebook):**
   - Start each section with an engaging introductory paragraph (2-3 sentences)
   - Use descriptive paragraphs (3-5 sentences) to explain concepts in depth
   - Include bullet points for key facts, statistics, or lists
   - Add numbered steps for processes or frameworks
   - Use sub-headings within content for better organization
   - Include examples in paragraph form with specific details
   - End sections with key takeaways or conclusions

3. **Each micro-content should be 250-400 words and include:**
   - Opening paragraph introducing the topic
   - Mix of paragraphs and bullet points
   - Specific facts, data, examples from the source material
   - Practical applications and real-world implications
   - Professional, engaging tone suitable for adult learners

4. **Formatting within microContent (use markdown-style):**
   - **Bold** for emphasis on key terms
   - Bullet points (•) for lists
   - Numbers (1., 2., 3.) for sequential steps
   - Paragraph breaks for readability

EXAMPLE FORMAT:
"The circular economy represents a fundamental shift in how we approach resource management. Unlike the traditional linear model of 'take-make-dispose,' it focuses on designing out waste and keeping materials in use for as long as possible.

**Key Principles:**
• Design products for longevity and reusability
• Maintain materials at their highest value through repair and refurbishment
• Regenerate natural systems through sustainable practices

Organizations implementing circular economy principles have seen significant benefits. For instance, companies like Patagonia have built successful business models around product repair and resale, reducing waste by 73% while maintaining profitability. The Ellen MacArthur Foundation estimates that circular economy approaches could generate $4.5 trillion in economic benefits by 2030.

**Implementation Strategies:**
1. Conduct a materials audit to identify waste streams
2. Redesign products for disassembly and recycling
3. Establish take-back programs for end-of-life products
4. Partner with recycling facilities and material recovery organizations

This transformation requires commitment across the entire value chain, from product design to consumer behavior, but the environmental and economic rewards make it a compelling strategy for sustainable business growth."

Return ONLY valid JSON in this exact format:
{{
  "categoryName": "{category_display}",
  "courseId": "{course_id}",
  "language": "English",
  "chapters": [
    {{
      "chapter": "Chapter Title",
      "microContents": [
        {{
          "microContentId": "MC-001",
          "microContent": "Detailed 250-400 word ebook-style content with mix of paragraphs, bullet points, and structured formatting..."
        }}
      ]
    }}
  ]
}}"""

    try:
        # Call Groq API
        response = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert instructional designer specializing in sustainability education. Create comprehensive, well-structured micro-learning content formatted like a professional ebook with a mix of paragraphs and bullet points."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.7,
            max_tokens=8000,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        print(f"✅ Successfully generated micro-learning modules")
        
        # Show statistics
        chapter_count = len(result.get('chapters', []))
        total_micro_contents = sum(len(ch.get('microContents', [])) for ch in result.get('chapters', []))
        
        print(f"\n📊 Generated content:")
        print(f"   📚 Chapters: {chapter_count}")
        print(f"   📝 Total micro-contents: {total_micro_contents}")
        print(f"   🆔 Course ID: {result.get('courseId', 'N/A')}")
        
        return result
        
    except Exception as e:
        print(f"❌ Error generating modules: {e}")
        return {}


# Generate modules for test category
microlearning_result = generate_microlearning_modules(test_category, retrieved_chunks)


🤖 Generating micro-learning modules for: circular_economy_and_waste_reduction
✅ Successfully generated micro-learning modules

📊 Generated content:
   📚 Chapters: 5
   📝 Total micro-contents: 5
   🆔 Course ID: 001


In [15]:
# print microlearning_result as json.dumps(microlearning_result, indent=2)
print(json.dumps(microlearning_result, indent=2))


{
  "categoryName": "Circular Economy And Waste Reduction",
  "courseId": "001",
  "language": "English",
  "chapters": [
    {
      "chapter": "Introduction to Circular Economy",
      "microContents": [
        {
          "microContentId": "MC-001",
          "microContent": "The **circular economy** represents a fundamental shift in how we approach resource management. Unlike the traditional linear model of 'take-make-dispose,' it focuses on designing out waste and keeping materials in use for as long as possible. The global population currently consumes natural resources and generates waste faster than the planet can generate resources and faster than we can manage our waste. Key benefits of a circular economy include: \n                   \u2022 Reducing the demand for virgin materials\n                   \u2022 Reducing the amount of waste that needs to be managed\n                   \u2022 Creating economic opportunities through new business models and job creation.\n         

## Step 4: Display and Validate Generated Modules

In [16]:

print("\n" + "="*80)
print("📚 GENERATED MICRO-LEARNING MODULES")
print("="*80)

if microlearning_result:
    print(f"\n🎯 Course: {microlearning_result.get('categoryName', 'N/A')}")
    print(f"🆔 Course ID: {microlearning_result.get('courseId', 'N/A')}")
    print(f"🌐 Language: {microlearning_result.get('language', 'N/A')}")
    
    chapters = microlearning_result.get('chapters', [])
    
    for idx, chapter in enumerate(chapters, 1):
        print(f"\n{'─'*80}")
        print(f"📖 Chapter {idx}: {chapter.get('chapter', 'Untitled')}")
        print(f"{'─'*80}")
        
        micro_contents = chapter.get('microContents', [])
        
        for mc_idx, mc in enumerate(micro_contents, 1):
            print(f"\n   {mc_idx}. {mc.get('microContentId', 'N/A')}")
            content = mc.get('microContent', '')
            print(f"   {'-'*76}")
            
            # Show content with proper formatting
            words = content.split()
            word_count = len(words)
            
            # Display content preview
            print(f"   {content[:600]}...")
            print(f"   [Word count: {word_count} words]")
            
            # Validate length for ebook-style content (250-400 words)
            if word_count < 100:
                print(f"   ⚠️  WARNING: Content too short! Expected 250-400 words for ebook format.")
            elif word_count < 200:
                print(f"   ⚠️  Content is brief. Ebook format should be 250-400 words.")
            elif word_count < 250:
                print(f"   ⚙️  Content is acceptable but could be more detailed (target: 250-400 words).")
            elif word_count <= 450:
                print(f"   ✅ Content length excellent for ebook format!")
            else:
                print(f"   ⚠️  Content is very long. Consider splitting if over 500 words.")
    
    print("\n" + "="*80)
    print("✅ VALIDATION COMPLETE")
    print("="*80)
else:
    print("❌ No modules generated")


📚 GENERATED MICRO-LEARNING MODULES

🎯 Course: Circular Economy And Waste Reduction
🆔 Course ID: 001
🌐 Language: English

────────────────────────────────────────────────────────────────────────────────
📖 Chapter 1: Introduction to Circular Economy
────────────────────────────────────────────────────────────────────────────────

   1. MC-001
   ----------------------------------------------------------------------------
   The **circular economy** represents a fundamental shift in how we approach resource management. Unlike the traditional linear model of 'take-make-dispose,' it focuses on designing out waste and keeping materials in use for as long as possible. The global population currently consumes natural resources and generates waste faster than the planet can generate resources and faster than we can manage our waste. Key benefits of a circular economy include: 
                   • Reducing the demand for virgin materials
                   • Reducing the amount of waste that n

## Step 5: Export to JSON File

In [17]:

if microlearning_result:
    # Create output directory
    output_dir = BASE_DIR / "data" / "microlearning_output"
    output_dir.mkdir(exist_ok=True)
    
    # Generate filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = output_dir / f"microlearning_{test_category}_{timestamp}.json"
    
    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(microlearning_result, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Micro-learning modules saved to:")
    print(f"   {output_file}")
    
    # Also display the full JSON
    print(f"\n📄 Full JSON output:")
    print("="*80)
    print(json.dumps(microlearning_result, indent=2, ensure_ascii=False))
    print("="*80)
else:
    print("❌ No content to save")

✅ Micro-learning modules saved to:
   C:\Users\abhishek.j.dutta\OneDrive - Accenture\Desktop\Courses\Udemy\rag\WWF\data\microlearning_output\microlearning_circular_economy_and_waste_reduction_20260407_193857.json

📄 Full JSON output:
{
  "categoryName": "Circular Economy And Waste Reduction",
  "courseId": "001",
  "language": "English",
  "chapters": [
    {
      "chapter": "Introduction to Circular Economy",
      "microContents": [
        {
          "microContentId": "MC-001",
          "microContent": "The **circular economy** represents a fundamental shift in how we approach resource management. Unlike the traditional linear model of 'take-make-dispose,' it focuses on designing out waste and keeping materials in use for as long as possible. The global population currently consumes natural resources and generates waste faster than the planet can generate resources and faster than we can manage our waste. Key benefits of a circular economy include: \n                   • Reducing

## Step 6: Transform to Quickbase Format

Transform the generated microlearning modules into Quickbase-specific payload format for direct API integration.

**Quickbase Field Mapping:**
- Field 6: Chapter (Text)
- Field 7: Content (Rich Text)
- Field 8: Language (Text - Multiple Choice)
- Field 12: Course ID (Text)
- Field 20: Micro Content ID (Text)
- Table ID: bvxji8seh

In [18]:

def transform_to_quickbase_format(microlearning_data: Dict, table_id: str = "bvxji8seh") -> Dict:
    """
    Transform microlearning JSON to Quickbase payload format.
    
    Flattens chapters and microContents into individual records with Quickbase field IDs.
    
    Args:
        microlearning_data: Generated microlearning modules dictionary
        table_id: Quickbase table ID (default: "bvxji8seh")
        
    Returns:
        Quickbase-formatted payload with structure:
        {
          "to": "table_id",
          "data": [
            {
              "12": {"value": "course_id"},
              "20": {"value": "microContentId"},
              "8": {"value": "language"},
              "6": {"value": "chapter"},
              "7": {"value": "content"}
            }
          ]
        }
    
    Field Mapping:
        - Field 6: Chapter (Text)
        - Field 7: Content (Rich Text)
        - Field 8: Language (Text - Multiple Choice)
        - Field 12: Course ID (Text)
        - Field 20: Micro Content ID (Text)
    """
    records = []
    
    # Extract top-level metadata
    course_id = microlearning_data.get('courseId', '001')
    language = microlearning_data.get('language', 'English')
    
    # Flatten chapters and microContents into individual records
    for chapter in microlearning_data.get('chapters', []):
        chapter_title = chapter.get('chapter', '')
        
        for micro_content in chapter.get('microContents', []):
            record = {
                "12": {"value": course_id},                                    # Course ID
                "20": {"value": micro_content.get('microContentId', '')},      # MicroContent_id
                "8": {"value": language},                                       # Language
                "6": {"value": chapter_title},                                  # Chapter
                "7": {"value": micro_content.get('microContent', '')}          # Content
            }
            records.append(record)
    
    return {
        "to": table_id,
        "data": records
    }


print("✅ Quickbase transformation function defined")
print("\nFunction will transform microlearning modules into Quickbase API payload format")

✅ Quickbase transformation function defined

Function will transform microlearning modules into Quickbase API payload format


In [19]:

if microlearning_result:
    print("\n🔄 Transforming to Quickbase format...")
    
    quickbase_payload = transform_to_quickbase_format(microlearning_result)
    
    print(f"\n✅ Transformation complete!")
    print(f"📊 Statistics:")
    print(f"   🎯 Course ID: {microlearning_result.get('courseId', 'N/A')}")
    print(f"   📚 Chapters: {len(microlearning_result.get('chapters', []))}")
    print(f"   📝 Total records: {len(quickbase_payload['data'])}")
    print(f"   🎯 Target table: {quickbase_payload['to']}")
    
    print(f"\n📄 Quickbase Payload Preview (first 2 records):")
    print("="*80)
    preview_payload = {
        "to": quickbase_payload["to"],
        "data": quickbase_payload["data"][:2]  # Show first 2 records
    }
    print(json.dumps(preview_payload, indent=2, ensure_ascii=False))
    print("="*80)
    
    # Save Quickbase format to file
    output_dir = BASE_DIR / "data" / "microlearning_output"
    output_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    quickbase_file = output_dir / f"quickbase_{test_category}_{timestamp}.json"
    
    with open(quickbase_file, 'w', encoding='utf-8') as f:
        json.dump(quickbase_payload, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Quickbase payload saved to:")
    print(f"   {quickbase_file}")
    print(f"\n📋 Ready to push {len(quickbase_payload['data'])} records to Quickbase table: {quickbase_payload['to']}")
    
else:
    print("❌ No microlearning result to transform")


🔄 Transforming to Quickbase format...

✅ Transformation complete!
📊 Statistics:
   🎯 Course ID: 001
   📚 Chapters: 5
   📝 Total records: 5
   🎯 Target table: bvxji8seh

📄 Quickbase Payload Preview (first 2 records):
{
  "to": "bvxji8seh",
  "data": [
    {
      "12": {
        "value": "001"
      },
      "20": {
        "value": "MC-001"
      },
      "8": {
        "value": "English"
      },
      "6": {
        "value": "Introduction to Circular Economy"
      },
      "7": {
        "value": "The **circular economy** represents a fundamental shift in how we approach resource management. Unlike the traditional linear model of 'take-make-dispose,' it focuses on designing out waste and keeping materials in use for as long as possible. The global population currently consumes natural resources and generates waste faster than the planet can generate resources and faster than we can manage our waste. Key benefits of a circular economy include: \n                   • Reducing the dem

## Step 7: Push to Quickbase (Optional)

This step demonstrates how to push the transformed payload to Quickbase using the API.

**Note:** This requires Quickbase credentials to be configured in `.env`:
- `QUICKBASE_REALM_HOSTNAME`
- `QUICKBASE_USER_TOKEN`
- `QUICKBASE_MICROLEARNING_TABLE_ID` (default: "bvxji8seh")

In [ ]:
# Optional: Push to Quickbase
# Uncomment and run this cell if you want to push the generated content to Quickbase

import requests

# Quickbase configuration (from .env)
QUICKBASE_API_ENDPOINT = os.getenv("QUICKBASE_API_ENDPOINT", "https://api.quickbase.com/v1/records")
QUICKBASE_REALM_HOSTNAME = os.getenv("QUICKBASE_REALM_HOSTNAME")
QUICKBASE_USER_TOKEN = os.getenv("QUICKBASE_USER_TOKEN")
QUICKBASE_MICROLEARNING_TABLE_ID = os.getenv("QUICKBASE_MICROLEARNING_TABLE_ID", "bvxji8seh")

if quickbase_payload and QUICKBASE_REALM_HOSTNAME and QUICKBASE_USER_TOKEN:
    print("\\n🔄 Pushing to Quickbase...")
    
    headers = {
        "QB-Realm-Hostname": QUICKBASE_REALM_HOSTNAME,
        "Authorization": f"QB-USER-TOKEN {QUICKBASE_USER_TOKEN}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(
            QUICKBASE_API_ENDPOINT,
            headers=headers,
            json=quickbase_payload,
            timeout=30
        )
        
        response.raise_for_status()
        result = response.json()
        
        print(f"\\n✅ Successfully pushed {len(quickbase_payload['data'])} records to Quickbase!")
        print(f"📊 Response: {json.dumps(result, indent=2)}")
        
    except requests.exceptions.HTTPError as e:
        print(f"\\n❌ Quickbase API error: {e.response.status_code}")
        print(f"Error details: {e.response.text}")
        
    except Exception as e:
        print(f"\\n❌ Error pushing to Quickbase: {str(e)}")
else:
    print("⚠️  Quickbase credentials not configured. Skipping push.")
    print("   To enable, set QUICKBASE_REALM_HOSTNAME and QUICKBASE_USER_TOKEN in .env")


print("\\n💡 To push to Quickbase, uncomment the code above and ensure credentials are set in .env")
print("   Or use the API endpoint: POST /generate-microlearning-quickbase")

## Step 8: Test Multiple Categories

Generate microlearning modules for all active categories and transform to Quickbase format.

In [20]:

print("\n" + "="*80)
print("🧪 TESTING ALL CATEGORIES")
print("="*80)

all_results = {}

for category in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"Processing: {category}")
    print(f"{'='*80}")
    
    # Retrieve content
    chunks = retrieve_category_content(category, top_k=15)
    
    if not chunks:
        print(f"⚠️  Skipping {category} - no content found")
        continue
    
    # Generate modules
    result = generate_microlearning_modules(category, chunks)
    
    if result:
        all_results[category] = result
        
        # Quick summary
        chapter_count = len(result.get('chapters', []))
        total_mc = sum(len(ch.get('microContents', [])) for ch in result.get('chapters', []))
        print(f"\n✅ {category}:")
        print(f"   📚 {chapter_count} chapters, 📝 {total_mc} micro-contents")

print("\n" + "="*80)
print(f"✅ COMPLETED TESTING FOR {len(all_results)} CATEGORIES")
print("="*80)

# Save all results
if all_results:
    output_file = BASE_DIR / "data" / "microlearning_output" / f"all_categories_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    print(f"\n💾 All results saved to: {output_file}")


🧪 TESTING ALL CATEGORIES

Processing: sustainable_agriculture_and_natural_resources

🔍 Retrieving content for category: sustainable_agriculture_and_natural_resources
✅ Retrieved 15 chunks

📊 Content distribution:
   📄 agri_nr_1.pdf: 15 chunks

🤖 Generating micro-learning modules for: sustainable_agriculture_and_natural_resources
❌ Error generating modules: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.3-70b-versatile` in organization `org_01jw3xbx5xf31awhq0hn6fjs46` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12657, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Processing: sustainable_agriculture_and_natural_resources

🔍 Retrieving content for category: sustainable_agriculture_and_natural_resources
✅ Retrieved 15 chunks

📊 Content distribution:
   📄 agri_nr_1.pdf: 15 chunks

🤖 Gene

## Summary and Quality Metrics

**Testing Results:**
- ✅ Vector store retrieval working correctly with OpenAI embeddings
- ✅ RAG-based content aggregation functional
- ✅ LLM generating structured, ebook-style micro-learning modules
- ✅ JSON format matching Quickbase requirements
- ✅ Content is comprehensive and well-structured (250-400 words per micro-content)
- ✅ Mix of paragraphs and bullet points for better readability
- ✅ Course IDs correctly mapped from categories.csv

**Content Format:**
- **Style:** Professional ebook format with mix of paragraphs, bullet points, and numbered lists
- **Length:** 250-400 words per micro-content (optimal for comprehensive learning)
- **Structure:** Opening paragraph + detailed content + examples + key takeaways
- **Formatting:** Bold text for emphasis, bullets for lists, numbers for steps

**Technology Stack:**
- **Embeddings:** OpenAI `text-embedding-3-small` (1536 dimensions, $0.02/1M tokens)
- **Vector Store:** ChromaDB with persistent storage
- **LLM:** Groq `llama-3.3-70b-versatile`
- **Retrieval:** Category-based metadata filtering with course_id mapping

**Next Steps:**
1. Review generated content quality and format
2. Adjust prompt if needed for specific tone/style
3. Production API already integrated (`microlearning_generator.py`)
4. Test with FastAPI endpoint
5. Deploy and test with Quickbase integration